# Season Distribution Reshape

## Input
`mosquito_suitability.csv`: ERA5 climate normals, one row per city × month (1,423 cities)

## Output


`season_distribution_by_species.csv`  Long: one row per city × species × threshold | Grouped bar charts
`season_distribution_wide.csv`  Wide: one row per season length (Moderate only) | Side-by-side histograms

## Logic
1. For each city and each threshold (Early 0.2 / Moderate 0.3 / Strict 0.4), count months where suitability ≥ threshold
2. Output long format with both species → `season_distribution_by_species.csv`
3. Pivot Moderate threshold to wide format → `season_distribution_wide.csv`

In [ ]:
import pandas as pd
df = pd.read_csv('mosquito_suitability.csv')

results = []

for threshold_name, threshold_val in [('Early', 0.2), ('Moderate', 0.3), ('Strict', 0.4)]:
    for (city, country), group in df.groupby(['city', 'country']):
        season_aegypti = (group['suitability_score_aegypti'] >= threshold_val).sum()
        season_albopictus = (group['suitability_score_albopictus'] >= threshold_val).sum()

        results.append({'city': city, 'country': country,
                        'species': 'Ae. aegypti', 'season_length': int(season_aegypti),
                        'threshold': threshold_name})
        results.append({'city': city, 'country': country,
                        'species': 'Ae. albopictus', 'season_length': int(season_albopictus),
                        'threshold': threshold_name})

out = pd.DataFrame(results)
out.to_csv('season_distribution_by_species.csv', index=False)
print(out.shape)
print(out.head(6))


(8538, 5)
       city  country         species  season_length threshold
0       Aba  Nigeria     Ae. aegypti             12     Early
1       Aba  Nigeria  Ae. albopictus             12     Early
2  Abeokuta  Nigeria     Ae. aegypti             12     Early
3  Abeokuta  Nigeria  Ae. albopictus             12     Early
4   Abhepur    India     Ae. aegypti              5     Early
5   Abhepur    India  Ae. albopictus              4     Early


In [ ]:
import pandas as pd
df = pd.read_csv('season_distribution_by_species.csv')

df_mod = df[df['threshold'] == 'Moderate']

pivot = df_mod.groupby(['season_length', 'species'])['city'].nunique().unstack()
pivot.columns = ['Ae. aegypti', 'Ae. albopictus']
pivot = pivot.reset_index()
pivot['season_length'] = pivot['season_length'].astype(str)
pivot.to_csv('season_distribution_wide.csv', index=False)
print(pivot)


   season_length  Ae. aegypti  Ae. albopictus
0              0           86              42
1              1           37              33
2              2           75              80
3              3          145             145
4              4          131             513
5              5          249             128
6              6          164              33
7              7           76              29
8              8           79              37
9              9           59              45
10            10           29              20
11            11           25              17
12            12          268             301
